In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
from scipy.stats import skew
warnings.filterwarnings('ignore')

DELIVERY_DIR = Path("../../data/raw/delivery")
print("Files:", [f.name for f in DELIVERY_DIR.iterdir()])

sample = pd.read_csv(DELIVERY_DIR / "delivery_sh.csv", nrows=5)
print(f"\nColumns: {list(sample.columns)}")
print(f"\nSample:")
print(sample.to_string())

Files: ['delivery_sh.csv', 'delivery_jl.csv', 'delivery_cq.csv', 'delivery_yt.csv', 'delivery_hz.csv']

Columns: ['order_id', 'region_id', 'city', 'courier_id', 'lng', 'lat', 'aoi_id', 'aoi_type', 'accept_time', 'accept_gps_time', 'accept_gps_lng', 'accept_gps_lat', 'delivery_time', 'delivery_gps_time', 'delivery_gps_lng', 'delivery_gps_lat', 'ds']

Sample:
   order_id  region_id      city  courier_id        lng       lat  aoi_id  aoi_type     accept_time accept_gps_time  accept_gps_lng  accept_gps_lat   delivery_time delivery_gps_time  delivery_gps_lng  delivery_gps_lat   ds
0   3158819          1  Shanghai         164  121.52128  31.06614     450         1  06-04 11:05:00  06-04 11:05:00       121.52281        31.10598  06-04 17:40:00    06-04 17:40:00         121.52407          31.06614  604
1    751342          1  Shanghai         164  121.52124  31.06687     450         1  06-04 11:18:00  06-04 11:18:00       121.52286        31.10599  06-04 15:06:00    06-04 15:06:00         121.

In [2]:
CITY_FILES = {
    "Shanghai": "delivery_sh.csv", "Hangzhou": "delivery_hz.csv",
    "Chongqing": "delivery_cq.csv", "Yantai": "delivery_yt.csv",
    "Jilin": "delivery_jl.csv",
}

frames = []
for city, fname in CITY_FILES.items():
    d = pd.read_csv(DELIVERY_DIR / fname, low_memory=False)
    print(f"{city:12s} → {len(d):>10,} rows")
    frames.append(d)

df = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(df):,} rows")

print("\nNull counts in key columns:")
print(df[['accept_time','delivery_time','accept_gps_lng','accept_gps_lat',
          'delivery_gps_lng','delivery_gps_lat']].isnull().sum())

Shanghai     →  1,483,864 rows
Hangzhou     →  1,861,600 rows
Chongqing    →    931,351 rows
Yantai       →    206,431 rows
Jilin        →     31,415 rows

Total: 4,514,661 rows

Null counts in key columns:
accept_time            0
delivery_time          0
accept_gps_lng      3377
accept_gps_lat      3377
delivery_gps_lng       0
delivery_gps_lat       0
dtype: int64


In [ ]:
for col in ['accept_time', 'delivery_time']:
    df[col] = pd.to_datetime(df[col], format='%m-%d %H:%M:%S', errors='coerce')

print("Parse nulls (NaT):")
print(df[['accept_time','delivery_time']].isnull().sum())

df['delivery_duration_min'] = (df['delivery_time'] - df['accept_time']).dt.total_seconds() / 60

print("\ndelivery_duration_min distribution:")
print(df['delivery_duration_min'].describe(percentiles=[.01,.25,.5,.75,.95,.99]).to_string())

print(f"\nNegative durations (delivery before accept!): {(df['delivery_duration_min'] < 0).sum():,}")
print(f"Zero durations: {(df['delivery_duration_min'] == 0).sum():,}")
print(f"Over 24 hours: {(df['delivery_duration_min'] > 1440).sum():,}")

Parse nulls (NaT):
accept_time      0
delivery_time    0
dtype: int64

delivery_duration_min distribution:
count    4.514661e+06
mean     1.858257e+02
std      7.801143e+02
min     -4.214170e+05
1%       3.000000e+00
25%      5.600000e+01
50%      1.050000e+02
75%      1.930000e+02
95%      4.830000e+02
99%      1.521000e+03
max      1.664990e+05

Negative durations (delivery before accept!): 3
Zero durations: 12,930
Over 24 hours: 48,590


In [ ]:
print("Negative duration rows (should be impossible):")
print(df[df['delivery_duration_min'] < 0][['accept_time','delivery_time','delivery_duration_min','city']].to_string())

corrupt = (df['delivery_duration_min'] < 0) | (df['delivery_duration_min'] > 10080)  # >7 days
print(f"\nCorrupt (negative or >7 days): {corrupt.sum():,} ({corrupt.mean()*100:.3f}%)")
print(f"Zero-duration (keep as anomaly candidates): {(df['delivery_duration_min']==0).sum():,}")

Negative duration rows (should be impossible):
                accept_time       delivery_time  delivery_duration_min       city
3785762 1900-10-31 10:57:00 1900-01-11 19:20:00              -421417.0  Chongqing
3863874 1900-10-31 12:14:00 1900-01-13 14:16:00              -418918.0  Chongqing
3868023 1900-10-22 17:54:00 1900-01-19 12:56:00              -397738.0  Chongqing

Corrupt (negative or >7 days): 1,804 (0.040%)
Zero-duration (keep as anomaly candidates): 12,930


In [ ]:
before = len(df)
df = df[(df['delivery_duration_min'] >= 0) & (df['delivery_duration_min'] <= 10080)].copy()
after = len(df)
print(f"Removed {before-after:,} corrupt rows ({(before-after)/before*100:.3f}%)")
print(f"Remaining: {after:,}")

print("\ndelivery_duration_min after cleaning:")
print(df['delivery_duration_min'].describe(percentiles=[.01,.25,.5,.75,.95,.99]).to_string())

Removed 1,804 corrupt rows (0.040%)
Remaining: 4,512,857

delivery_duration_min after cleaning:
count    4.512857e+06
mean     1.769550e+02
std      3.633508e+02
min      0.000000e+00
1%       3.000000e+00
25%      5.600000e+01
50%      1.050000e+02
75%      1.930000e+02
95%      4.820000e+02
99%      1.480000e+03
max      1.007900e+04


In [ ]:
print(f"Duplicate order_ids: {df['order_id'].duplicated().sum():,}")

print(f"accept_gps nulls: {df['accept_gps_lng'].isnull().sum():,}")
print(f"delivery_gps nulls: {df['delivery_gps_lng'].isnull().sum():,}")

print("\nGPS coordinate ranges:")
for c in ['accept_gps_lng','accept_gps_lat','delivery_gps_lng','delivery_gps_lat','lng','lat']:
    print(f"  {c:20s} min={df[c].min():.2f}  max={df[c].max():.2f}")

print(f"\nZero delivery_gps_lng: {(df['delivery_gps_lng']==0).sum():,}")
print(f"Zero accept_gps_lng: {(df['accept_gps_lng']==0).sum():,}")

Duplicate order_ids: 0
accept_gps nulls: 3,376
delivery_gps nulls: 0

GPS coordinate ranges:
  accept_gps_lng       min=-0.00  max=126.63
  accept_gps_lat       min=-0.00  max=43.95
  delivery_gps_lng     min=-0.00  max=139.76
  delivery_gps_lat     min=-0.00  max=45.76
  lng                  min=102.08  max=126.82
  lat                  min=23.11  max=44.22

Zero delivery_gps_lng: 18
Zero accept_gps_lng: 14


In [ ]:
before = len(df)
gps_corrupt = (
    (df['accept_gps_lng'].abs() < 1) | (df['accept_gps_lat'].abs() < 1) |
    (df['delivery_gps_lng'].abs() < 1) | (df['delivery_gps_lat'].abs() < 1)
)
print(f"Rows with corrupt (near-zero) GPS: {gps_corrupt.sum():,}")

df = df[~gps_corrupt].copy()
print(f"Removed {before-len(df):,} → {len(df):,} remaining")

print("\nGPS ranges after cleaning:")
for c in ['accept_gps_lng','accept_gps_lat','delivery_gps_lng','delivery_gps_lat']:
    print(f"  {c:20s} min={df[c].min():.2f}  max={df[c].max():.2f}")

Rows with corrupt (near-zero) GPS: 284
Removed 284 → 4,512,573 remaining

GPS ranges after cleaning:
  accept_gps_lng       min=80.12  max=126.63
  accept_gps_lat       min=29.04  max=43.95
  delivery_gps_lng     min=101.72  max=139.76
  delivery_gps_lat     min=22.36  max=45.76


In [8]:
df.head()

,order_id,region_id,city,courier_id,lng,lat,aoi_id,aoi_type,accept_time,accept_gps_time,accept_gps_lng,accept_gps_lat,delivery_time,delivery_gps_time,delivery_gps_lng,delivery_gps_lat,ds,delivery_duration_min
0,3158819,1,Shanghai,164,121.52128,31.06614,450,1,1900-06-04 11:05:00,06-04 11:05:00,121.52281,31.10598,1900-06-04 17:40:00,06-04 17:40:00,121.52407,31.06614,604,395.0
1,751342,1,Shanghai,164,121.52124,31.06687,450,1,1900-06-04 11:18:00,06-04 11:18:00,121.52286,31.10599,1900-06-04 15:06:00,06-04 15:06:00,121.52412,31.06618,604,228.0
2,3380476,1,Shanghai,164,121.52106,31.06731,450,1,1900-06-03 10:13:00,06-03 10:13:00,121.52285,31.10591,1900-06-03 15:11:00,06-03 15:11:00,121.52059,31.06672,603,298.0
3,2184571,1,Shanghai,164,121.52128,31.06616,450,1,1900-06-04 10:39:00,06-04 10:39:00,121.52282,31.10593,1900-06-04 15:41:00,06-04 15:41:00,121.52280,31.10542,604,302.0
4,941371,1,Shanghai,164,121.52123,31.06614,450,1,1900-06-04 11:18:00,06-04 11:18:00,121.52285,31.10593,1900-06-04 14:07:00,06-04 14:07:00,121.52290,31.06758,604,169.0


In [ ]:
print("1. region_id vs city:")
print(df.groupby('city')['region_id'].unique() if 'region_id' in df.columns else "region_id already dropped")

print(f"\n2. Duplicate order_ids: {df['order_id'].duplicated().sum():,}")

print(f"\n3. ds range: {df['ds'].min()} → {df['ds'].max()}  ({df['ds'].nunique()} unique days)")

print(f"\n4. Rows per city:")
print(df['city'].value_counts())

print(f"\n5. Null counts (all columns):")
print(df.isnull().sum())

print(f"\n6. aoi_type unique: {sorted(df['aoi_type'].unique())}")

print(f"\n7. duration_min: min={df['delivery_duration_min'].min():.0f} max={df['delivery_duration_min'].max():.0f}")

print(f"\n8. Dtypes:")
print(df.dtypes)

1. region_id vs city:
city
Chongqing    [10, 11, 15, 24, 40, 48, 67, 70, 74, 103, 104,...
Hangzhou     [0, 3, 4, 12, 14, 19, 23, 26, 27, 28, 30, 34, ...
Jilin                                       [31, 53, 129, 165]
Shanghai     [1, 2, 5, 6, 7, 8, 9, 13, 16, 17, 18, 20, 21, ...
Yantai       [86, 95, 102, 111, 115, 118, 128, 137, 140, 14...
Name: region_id, dtype: object

2. Duplicate order_ids: 0

3. ds range: 501 → 1031  (184 unique days)

4. Rows per city:
city
Hangzhou     1860866
Shanghai     1483689
Chongqing     930297
Yantai        206316
Jilin          31405
Name: count, dtype: int64

5. Null counts (all columns):
order_id                    0
region_id                   0
city                        0
courier_id                  0
lng                         0
lat                         0
aoi_id                      0
aoi_type                    0
accept_time                 0
accept_gps_time             0
accept_gps_lng           3375
accept_gps_lat           3375
delivery_t

In [ ]:
DROP = ['region_id', 'accept_gps_time', 'delivery_gps_time']
df = df.drop(columns=DROP)

OUT = "../../data/processed/delivery_clean.parquet"
df.to_parquet(OUT, index=False)
print(f"Saved {len(df):,} rows → {OUT}")
print(f"Columns: {list(df.columns)}")
print(f"\nFinal shape: {df.shape}")

Saved 4,512,573 rows → ../../data/processed/delivery_clean.parquet
Columns: ['order_id', 'city', 'courier_id', 'lng', 'lat', 'aoi_id', 'aoi_type', 'accept_time', 'accept_gps_lng', 'accept_gps_lat', 'delivery_time', 'delivery_gps_lng', 'delivery_gps_lat', 'ds', 'delivery_duration_min']

Final shape: (4512573, 15)


In [11]:
check = pd.read_parquet("../../data/processed/delivery_clean.parquet")
print(f"Rows: {len(check):,}")
print(f"Columns ({len(check.columns)}): {list(check.columns)}")
print(f"\nNull check:")
print(check.isnull().sum())
print(f"\nDuration range: {check['delivery_duration_min'].min():.0f} to {check['delivery_duration_min'].max():.0f}")
print(f"Duplicate order_ids: {check['order_id'].duplicated().sum()}")
print(f"ds range: {check['ds'].min()} to {check['ds'].max()}")

Rows: 4,512,573
Columns (15): ['order_id', 'city', 'courier_id', 'lng', 'lat', 'aoi_id', 'aoi_type', 'accept_time', 'accept_gps_lng', 'accept_gps_lat', 'delivery_time', 'delivery_gps_lng', 'delivery_gps_lat', 'ds', 'delivery_duration_min']

Null check:
order_id                    0
city                        0
courier_id                  0
lng                         0
lat                         0
aoi_id                      0
aoi_type                    0
accept_time                 0
accept_gps_lng           3375
accept_gps_lat           3375
delivery_time               0
delivery_gps_lng            0
delivery_gps_lat            0
ds                          0
delivery_duration_min       0
dtype: int64

Duration range: 0 to 10079
Duplicate order_ids: 0
ds range: 501 to 1031


In [ ]:
print("Per-city duration: skewness + percentile gaps (decides z-score vs robust scaling)")
print(f"{'city':12s} {'median':>8s} {'P95':>8s} {'P99':>8s} {'max':>8s} {'skew':>8s}")
for city in df['city'].unique():
    d = df[df['city']==city]['delivery_duration_min']
    print(f"{city:12s} {d.median():>8.0f} {d.quantile(.95):>8.0f} {d.quantile(.99):>8.0f} {d.max():>8.0f} {skew(d):>8.1f}")

print("\nDuration by aoi_type (building type):")
aoi_dur = df.groupby('aoi_type')['delivery_duration_min'].agg(['median','count'])
print(aoi_dur.round(0).to_string())

Per-city duration: skewness + percentile gaps (decides z-score vs robust scaling)
city           median      P95      P99      max     skew
Shanghai           72      322      651     9935     15.9
Hangzhou          115      422      761    10026     14.8
Chongqing         151      732     3184    10079      7.5
Yantai            208      597     2049     9274      6.3
Jilin             175      469      661     3573      2.0

Duration by aoi_type (building type):
          median    count
aoi_type                 
0           92.0   150848
1           98.0  2958949
2           94.0    68635
3          107.0    16665
4          246.0   109521
5           92.0     1615
6           99.0     6016
7          110.0    62714
8          128.0    14741
9          113.0    27553
10         138.0    13844
11         178.0     1279
12          99.0    26193
13         109.0     9090
14         127.0  1038103
15         307.0     6807
